<a href="https://colab.research.google.com/github/ibtihalalf/Sdaia-Bootcamp/blob/main/C4/M2/Ex4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# M2.Ex4 - ASR YouTube Task
# Use Case: YouTube Lecture Intelligence Assistant
# =========================

!pip install -qU langchain langchain-community youtube-transcript-api pytube yt-dlp "docling[asr]" openai

import os
import subprocess
from pathlib import Path

from langchain_community.document_loaders import YoutubeLoader
from langchain_community.document_loaders.youtube import TranscriptFormat

from docling.datamodel import asr_model_specs
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AsrPipelineOptions
from docling.document_converter import AudioFormatOption, DocumentConverter
from docling.pipeline.asr_pipeline import AsrPipeline

from openai import OpenAI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 26.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)


In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

In [4]:
def get_youtube_transcript(video_url):
    """
    First try normal YouTube transcript.
    This works if the video has captions/subtitles.
    """
    try:
        loader = YoutubeLoader.from_youtube_url(
            video_url,
            add_video_info=True,
            transcript_format=TranscriptFormat.CHUNKS,
            chunk_size_seconds=60,
            language=["en", "ar"],
            translation="en"
        )
        docs = loader.load()

        if len(docs) == 0:
            raise Exception("No transcript found.")

        print("Transcript loaded from YouTube captions.")
        return docs

    except Exception as e:
        print("No YouTube transcript found. Switching to Docling ASR...")
        print("Reason:", e)
        return None

In [5]:
def download_youtube_audio(video_url, output_path="youtube_audio.mp3"):
    """
    Download audio from YouTube using yt-dlp.
    """
    command = [
        "yt-dlp",
        "-x",
        "--audio-format", "mp3",
        "-o", output_path.replace(".mp3", ".%(ext)s"),
        video_url
    ]

    subprocess.run(command, check=True)
    return Path(output_path)

In [6]:
def transcribe_with_docling_asr(audio_path):
    """
    Use Docling ASR pipeline when YouTube captions are not available.
    """
    pipeline_options = AsrPipelineOptions()
    pipeline_options.asr_options = asr_model_specs.WHISPER_TURBO

    converter = DocumentConverter(
        format_options={
            InputFormat.AUDIO: AudioFormatOption(
                pipeline_cls=AsrPipeline,
                pipeline_options=pipeline_options,
            )
        }
    )

    result = converter.convert(audio_path)
    markdown_text = result.document.export_to_markdown()

    print("Transcript created using Docling ASR.")
    return markdown_text

In [7]:
def prepare_text_from_docs(docs):
    """
    Convert LangChain Documents into one text.
    """
    full_text = ""

    for doc in docs:
        timestamp_url = doc.metadata.get("source", "")
        content = doc.page_content
        full_text += f"\n\nSource: {timestamp_url}\n{content}"

    return full_text

In [8]:
def analyze_transcript(transcript_text):
    """
    Generate summary, key points, and quiz.
    """
    prompt = f"""
You are an AI learning assistant.

Analyze the following YouTube lecture transcript.

Return:
1. Short summary
2. Main topics
3. Important timestamps if available
4. 5 quiz questions with answers
5. Suggested title for notes

Transcript:
{transcript_text[:12000]}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful educational assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [10]:
# =========================
# Run the project
# =========================

video_url = "https://youtu.be/T-D1OfcDW1M?si=Cb9PS9uE-w0NoRx-"

docs = get_youtube_transcript(video_url)

if docs:
    transcript_text = prepare_text_from_docs(docs)
else:
    audio_path = download_youtube_audio(video_url)
    transcript_text = transcribe_with_docling_asr(audio_path)

result = analyze_transcript(transcript_text)

print(result)

No YouTube transcript found. Switching to Docling ASR...
Reason: HTTP Error 400: Bad Request


100%|█████████████████████████████████████| 1.51G/1.51G [00:41<00:00, 39.1MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:02.340]  Large language models, they are everywhere.
[00:02.660 --> 00:07.200]  They get some things amazingly right and other things very interestingly wrong.
[00:07.680 --> 00:09.240]  My name is Marina Donalewski.
[00:09.460 --> 00:13.600]  I am a senior research scientist here at IBM Research, and I want to tell you
[00:13.600 --> 00:18.000]  about a framework to help large language models be more accurate and more up to
[00:18.000 --> 00:21.760]  date, retrieval augmented generation or RAG.
[00:22.480 --> 00:24.640]  Let's just talk about the generation part for a minute.
[00:24.800 --> 00:26.400]  So forget the retrieval augmented.
[00:26.400 --> 00:31.860]  So the generation, this refers to large language models or LLMs that generate
[00:31.860 --> 00:35.380]  text in response to a user query referred to as a prompt.
[00:35.740 --> 00:37.960]